<a href="https://colab.research.google.com/github/Uzma-Iqbal88/Generative_AI_RAG/blob/main/Generative_AI_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth transformers accelerate bitsandbytes
!pip install sentence-transformers faiss-cpu PyPDF2


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [2]:
from google.colab import files
uploaded = files.upload()


Saving sample_rag_docs.pdf to sample_rag_docs.pdf


In [3]:
import PyPDF2

pdf = open("sample_rag_docs.pdf", "rb")
reader = PyPDF2.PdfReader(pdf)

documents = []

for page in reader.pages:
    documents.append(page.extract_text())

print(documents)


['Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially\ncomputer systems.\nMachine Learning (ML) is a subset of AI that enables systems to learn and improve from\nexperience without being explicitly programmed.\nDeep Learning is a specialized form of machine learning that uses neural networks with many\nlayers.\nNatural Language Processing (NLP) allows computers to understand, interpret, and generate\nhuman language.\nComputer Vision enables machines to interpret and make decisions based on visual data.\n']


In [4]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))


In [6]:
query = "What is Artificial Intelligence?"


In [7]:
query_embedding = embed_model.encode([query])

D, I = index.search(np.array(query_embedding), k=2)

retrieved_docs = [documents[i] for i in I[0]]

print("Retrieved Docs:", retrieved_docs)


Retrieved Docs: ['Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially\ncomputer systems.\nMachine Learning (ML) is a subset of AI that enables systems to learn and improve from\nexperience without being explicitly programmed.\nDeep Learning is a specialized form of machine learning that uses neural networks with many\nlayers.\nNatural Language Processing (NLP) allows computers to understand, interpret, and generate\nhuman language.\nComputer Vision enables machines to interpret and make decisions based on visual data.\n', 'Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially\ncomputer systems.\nMachine Learning (ML) is a subset of AI that enables systems to learn and improve from\nexperience without being explicitly programmed.\nDeep Learning is a specialized form of machine learning that uses neural networks with many\nlayers.\nNatural Language Processing (NLP) allows computers to unde

In [8]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True
)


/tmp/ipykernel_3183/446679148.py:1: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 🔹 1️⃣ Stable Model Load (Unsloth 4-bit issue fix)
model_name = "distilgpt2"  # Stable alternative
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

# 🔹 2️⃣ Short context to avoid sequence errors
context = " ".join(retrieved_docs)[:200]

# 🔹 3️⃣ Create prompt
prompt = f"Answer using this context:\n{context}\n\nQuestion: {query}"

# 🔹 4️⃣ Tokenize (move to GPU)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 🔹 5️⃣ Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

# 🔹 6️⃣ Decode and print
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Final Answer:\n", answer)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Final Answer:
 Answer using this context:
Artificial Intelligence (AI) is the simulation of human intelligence processes by machines, especially
computer systems.
Machine Learning (ML) is a subset of AI that enables systems to learn and impro

Question: What is Artificial Intelligence?
AI is a subset of AI that enables systems to learn and impro
Question: What is Artificial Intelligence?
AI is a subset of AI that enables systems to learn and impro
Question: What is Artificial Intelligence?
AI is a subset of AI that enables systems to learn and impro
Question: What is Artificial Intelligence?
AI is a subset of AI that enables systems to learn and impro
Question: What is Artificial Intelligence?
AI is a subset of AI that enables systems to learn
